# 03 — Activation Patching

Causal layer sweep: patch source residuals into a target run and plot
$\Delta P(\text{answer})$ vs start layer — then **recommend layers to swap**.

Docs: `docs/activation_patching_reimpl/`

In [ ]:
# --- Colab / local bootstrap (Drive + wheels + swappable model) ---
# Drive layout expected:
#   MyDrive/multilingual-mechinterp/
#     dist/*.whl
#     data/all200questions_persianMiddleEastCulture.json
#     configs/  notebooks/  results/
#
# Edit MODEL_NAME in notebooks/colab_setup.py (Qwen2.5 now; Gemma later),
# or override below after bootstrap.

from pathlib import Path
import runpy

def _resolve_setup_script() -> Path:
    here = Path.cwd()
    candidates = [
        here / "colab_setup.py",
        here / "notebooks" / "colab_setup.py",
        here.parent / "notebooks" / "colab_setup.py",
        Path("/content/drive/MyDrive/multilingual-mechinterp/notebooks/colab_setup.py"),
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(
        "colab_setup.py not found. Mount Drive with the project folder, "
        "or open the notebook from the repo."
    )

_setup = runpy.run_path(str(_resolve_setup_script()))
globals().update({k: _setup[k] for k in _setup["EXPORTS"]})

# Session overrides (uncomment as needed):
# MODEL_NAME = "google/gemma-2-2b"
# MODEL_TRUST_REMOTE_CODE = False
# USE_TINY_OFFLINE = True   # demos without downloading HF weights

import matplotlib.pyplot as plt
import torch

from multilingual_mechinterp.utils import ensure_dir, load_config

cfg_path = CONFIG_DIR / "qwen25.yaml"
cfg = load_config(cfg_path) if cfg_path.exists() else {}
if "model" in cfg and not USE_TINY_OFFLINE:
    # keep notebook MODEL_NAME as source of truth; cfg is fallback metadata
    pass

print("Ready.")
print(" ROOT =", ROOT)
print(" DATA =", DATA_DIR)
print(" DIST =", DIST_DIR)
print(" MODEL =", MODEL_NAME, "| tiny=", USE_TINY_OFFLINE)

from multilingual_mechinterp.patching import (
    TinyCausalLM, recommend_layers, run_patching,
)
from multilingual_mechinterp.data import culture_prompt_pairs, load_culture_questions

OUT = ensure_dir(RESULTS_DIR / "patching")


## Load experiment model

Uses `MODEL_NAME` from `colab_setup.py` (default **Qwen2.5**). Set `USE_TINY_OFFLINE=True` for demos without HF downloads.


In [ ]:
# Real model (Qwen now; change MODEL_NAME for Gemma later) OR tiny offline
# model = load_experiment_model()
# For gated Gemma: export HF_TOKEN=... or pass token=...

# Default path in analysis cells below uses tiny models for speed.
# Swap in `model = load_experiment_model()` when you are ready for Qwen/Gemma.
print("To load HF weights:", f"load_experiment_model({MODEL_NAME!r})")
print("Culture JSON:", culture_json_path(), "exists=", culture_json_path().exists())


## 1. Offline causal-effect curve

In [ ]:
model = TinyCausalLM(n_layers=8, d_model=32, seed=0)

source = "The capital of France is Paris"
target = "The capital of Italy is"
answer = "Paris"

result = run_patching(
    model,
    source_prompt=source,
    target_prompt=target,
    answer=answer,
    window=None,
)
layers = recommend_layers(result, top_k=3, min_effect=-1.0)
print("baseline P(answer)=", round(result.baseline_score, 4))
print("best_layer=", result.best_layer)
print("recommend_layers=", layers)

In [ ]:
xs = sorted(result.scores)
ys = [result.scores[L] for L in xs]
ps = [result.patched_probs[L] for L in xs]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(xs, ys, marker="o", color="#4C78A8")
axes[0].axhline(0, color="k", lw=0.8, alpha=0.4)
if result.best_layer is not None:
    axes[0].axvline(result.best_layer, color="#E45756", ls="--", label="best")
axes[0].set_xlabel("start layer")
axes[0].set_ylabel("ΔP(answer)")
axes[0].set_title("Causal effect curve")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(xs, ps, marker="o", color="#72B7B2", label="patched")
axes[1].axhline(result.baseline_score, color="#F58518", ls="--", label="baseline")
axes[1].set_xlabel("start layer")
axes[1].set_ylabel("P(answer)")
axes[1].set_title("Patched vs baseline probability")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(OUT / "causal_effect_curve.png", dpi=150)
plt.show()

## 2. Culture dataset pair (EN → FA)

Uses `data/all200questions_persianMiddleEastCulture.json` when present.

In [ ]:
from multilingual_mechinterp.data import culture_prompt_pairs, load_culture_questions

path = ROOT / "data" / "all200questions_persianMiddleEastCulture.json"
if path.exists():
    items = load_culture_questions(path, limit=1)
    pair = culture_prompt_pairs(items, source_lang="english", target_lang="persian", limit=1)[0]
    print(pair["source_answer"], "|", pair["target_answer"])
    print(pair["source_prompt"][:240], "...")

    # Toy model only demos the API; swap in load_model(...) for real effects
    cult = run_patching(
        model,
        source_prompt=pair["source_prompt"][:180],
        target_prompt=pair["target_prompt"][:180],
        answer=pair["source_answer"].split()[0],
    )
    print("culture best_layer=", cult.best_layer, "scores sample=", dict(list(cult.scores.items())[:4]))
else:
    print("Culture JSON not found at", path)

## 3. Real HF model (optional)

```python
from multilingual_mechinterp.models import load_model
model = load_model("EleutherAI/pythia-70m-deduped")
result = run_patching(model, en_prompt, fa_prompt, answer="Divan-e Hafez")
layers_to_swap = recommend_layers(result, top_k=3)
```